# Numpy 之一：创建数组

---

## 为什么需要 Numpy？

Python 原生的 list 做数值计算有两个致命问题：

```python
a = [1, 2, 3]
b = [4, 5, 6]
print(a + b)      # [1, 2, 3, 4, 5, 6]  拼接，不是相加！
print(a * 2)      # [1, 2, 3, 1, 2, 3]  重复，不是乘 2！
```

而且 list 里每个元素都是一个独立的 Python 对象，**内存开销大、计算慢**。

**Numpy 的 ndarray 解决了这两个问题**：

| | Python list | numpy ndarray |
|---|---|---|
| 元素类型 | 可以混装 | **必须统一**（这是快的前提） |
| 运算语义 | 拼接 / 重复 | **元素级**（`a + b` 是逐项相加） |
| 存储方式 | 一堆对象指针 | **一块连续内存** |
| 速度 | 慢 | 快几十倍 |

## 学习目标

1. 掌握 6 类创建方式
2. 分清 `zeros` / `ones` / `empty` 的区别
3. 理解 `*_like` 系列的作用
4. 会生成随机数组，并知道为什么实验里要设随机种子


In [ ]:
import numpy as np

print('numpy 版本：', np.__version__)


---
## 一、从列表 / 元组创建：最常用的入口


In [ ]:
list1 = [1, 2, 3]
arr = np.array(list1)
print('从列表：', arr, ' 类型：', type(arr))

print('从元组：', np.array((4, 5, 6)))
print('从嵌套列表（二维）：')
print(np.array([[1, 2, 3], [4, 5, 6]]))

# 【关键】指定 dtype 就决定了这块内存怎么用
print('\n整数数组：', np.array([1, 2, 3], dtype=int))
print('浮点数组：', np.array([1, 2, 3], dtype=float))
print('布尔数组：', np.array([0, 1, 2], dtype=bool))     # 非 0 都是 True

# 【对比】list 和 ndarray 的运算语义完全不同
py_list = [1, 2, 3]
np_arr = np.array([1, 2, 3])
print('\n列表 1+2：', py_list + [4, 5, 6], '  <- 拼接')
print('数组 1+2：', np_arr + np.array([4, 5, 6]), '  <- 逐项相加')


## 二、从已有数组创建：`copy` 和 `view` 的区别

这两个都能"复制"数组，但一个真复制，一个只是"换个角度看"。

| 方法 | 数据是否共享 | 改一个会影响另一个吗 |
|---|---|---|
| `arr.view()` | **共享** 同一块内存 | **会** |
| `arr.copy()` | 独立内存 | 不会 |

**类比**：`view` 像是给同一间房装了一面镜子（从别处看同一份东西），
`copy` 是照着原房又盖了一间。


In [ ]:
base = np.array([1, 2, 3])

v = base.view()
c = base.copy()

print('view 的数据源是 base 吗？', v.base is base)     # True —— 共享内存
print('copy 的数据源是 base 吗？', c.base is base)     # False —— 各自独立

base[0] = 999
print('\n改 base[0] 之后：')
print('  base =', base)
print('  view =', v, '  <- 跟着变了')
print('  copy =', c, '  <- 没变')


## 三、预定义形状的数组：`zeros` / `ones` / `eye` / `full`

**参数必须是一个"形状元组"**，注意写法：

```python
np.zeros((2, 3))    # 对：传一个元组
np.zeros(2, 3)      # 错：会被理解成两个参数
```


In [ ]:
print('全 0（2 行 3 列）：')
print(np.zeros((2, 3), dtype=int))

print('\n全 1（2 行 3 列）：')
print(np.ones((2, 3), dtype=int))

print('\n单位矩阵（主对角线为 1）：')
print(np.eye(3, dtype=int))

print('\n全定值（4 行 3 列，全是 5）：')
print(np.full((4, 3), 5, dtype=int))


### `np.empty()`：不初始化，内容是"内存里的垃圾值"

**它不是"全空"，而是"我不帮你清零"**。所以能看到随机数字。

**为什么会有这种反直觉的函数？** 因为清零要花时间。
如果你马上就会给每个位置赋值，那清零就是白费功夫。
**所以 `empty` 的正确用法是：先申请，再立刻填满。**


In [ ]:
e = np.empty((2, 3))
print('np.empty 的内容（是未初始化的垃圾值，每次可能不同）：')
print(e)

# 正确用法：申请完立刻填
e2 = np.empty((2, 3))
e2[:] = 7          # : 表示"所有位置"
print('\n立刻填满之后：')
print(e2)


## 四、`*_like` 系列：照着"别人"的形状造数组

当你要造一个和现有数组**形状完全相同**的容器（比如初始化一个累加器）时，用这个最省事。


In [ ]:
template = np.full((4, 3), 5, dtype=int)
print('模板数组形状：', template.shape)

zeros_like = np.zeros_like(template)
print('\nzeros_like -> 形状相同、内容全 0：')
print(zeros_like)

# 【重要】_like 系列既能复制形状，也能复制 dtype
print('   验证 dtype：', zeros_like.dtype, '（和模板的', template.dtype, '一致）')

# full_like：形状和类型照抄，值自己指定
six_like = np.full_like(template, 6)
print('\nfull_like(6) -> 形状相同、内容全 6：')
print(six_like)


## 五、等间隔 / 等差 / 等比数组

这三个是画图和数据采样时的主力，**区别必须记牢**：

| 函数 | 语义 | 关键参数 | 是否包含终点 |
|---|---|---|---|
| `arange(start, stop, step)` | 按**步长**生成 | step | **不包含** stop |
| `linspace(start, stop, num)` | 按**个数**均分 | num | **包含** stop |
| `logspace(start, stop, num, base)` | 按对数均分 | num, base | 包含 |


In [ ]:
# arange：从 0 开始，到 11 之前结束，步长 2
print('arange(0, 11, 2)     ->', np.arange(0, 11, 2))

# 只给一个参数时，从 0 开始
print('arange(5)            ->', np.arange(5))

# 浮点步长也可以用（但有精度陷阱）
print('arange(0, 1, 0.25)   ->', np.arange(0, 1, 0.25))

# linspace：从 0 到 10 之间均分 6 个点，包含 10
print('linspace(0, 10, 6)   ->', np.linspace(0, 10, 6))

# linspace 的第三个参数是"个数"，这点最容易和 arange 搞混
print('linspace(0, 10, 11)  ->', np.linspace(0, 10, 11))

# logspace：从 base^start 到 base^stop，取 num 个
print('logspace(0, 2, 3, base=2) ->', np.logspace(0, 2, 3, base=2))
# 2^0=1, 2^1=2, 2^2=4  ->  [1. 2. 4.]

# arange 的浮点陷阱：
print('\narange(0, 0.3, 0.1)  ->', np.arange(0, 0.3, 0.1))
print('最后一个数为什么不是 0.3？因为浮点数存不下精确的 0.1，累积误差导致 0.2+0.1=0.30000000000000004 > 0.3')
print('这种场景请改用 linspace：', np.linspace(0, 0.3, 4))


## 六、特殊矩阵（线性代数里常用）


In [ ]:
print('单位矩阵 np.eye(3)：')
print(np.eye(3, dtype=int))

print('\n对角矩阵 np.diag([1,2,3])（只保留对角线）：')
print(np.diag([1, 2, 3]))

print('\n对称矩阵（转置等于自己）：')
sym = np.array([[1, 2, 3], [2, 4, 5], [3, 5, 6]])
print(sym)
print('它的转置：')
print(sym.T)
print('两者相等吗？', np.array_equal(sym, sym.T))


## 七、随机数组

`np.random` 里有好几个函数，**区间开闭不一样**，很容易搞错。


In [ ]:
# 1) randint：整数，左闭右开 [low, high)
print('randint(0, 10, (3,3))：')
print(np.random.randint(0, 10, (3, 3)))

# 2) rand：均匀分布浮点，[0, 1)，参数直接写形状
print('\nrand(3, 3)：')
print(np.random.rand(3, 3))

# 3) uniform：指定区间的均匀分布浮点
print('\nuniform(0, 10, (3, 3))：')
print(np.random.uniform(0, 10, (3, 3)))

# 4) normal：正态分布，参数是 (均值, 标准差, 形状)
print('\nnormal(0, 1, (3, 3))：')
print(np.random.normal(0, 1, (3, 3)))

# 5) choice：从给定序列里抽样
print('\nchoice 从 1~10 抽 5 个：', np.random.choice(range(1, 11), 5))
print('choice 不重复抽 5 个：', np.random.choice(range(1, 11), 5, replace=False))


### 随机种子：让"随机"变得可复现

做实验、写教程、调试代码时，**结果每次都不一样**会让人抓狂。
`np.random.seed(n)` 就是给随机数生成器设定一个"起点"，
同一个种子 → 同一串随机数。


In [ ]:
np.random.seed(20)
a = np.random.randint(0, 10, (2, 5))
print('种子 20 第一次：')
print(a)

np.random.seed(20)
b = np.random.randint(0, 10, (2, 5))
print('\n种子 20 第二次（结果完全相同）：')
print(b)
print('\n两次结果一样吗？', np.array_equal(a, b))

np.random.seed(21)
c = np.random.randint(0, 10, (2, 5))
print('\n换个种子 21（结果就变了）：')
print(c)


> **说明**：`np.random.seed()` 属于"老接口"，现在更推荐用 `np.random.default_rng(seed)` 生成器对象。
> 但老接口在大量教程和代码里还在用，所以两种都要认识：
>
> ```python
> rng = np.random.default_rng(42)      # 新接口
> rng.integers(0, 10, (2, 5))
> ```


In [ ]:
rng = np.random.default_rng(42)
print('新接口 default_rng(42)：')
print(rng.integers(0, 10, (2, 5)))
print('再生成一次（同一个 rng 对象，会继续往下取）：')
print(rng.integers(0, 10, (2, 5)))


---
## 八、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| `np.zeros(2, 3)` | `TypeError` | 形状要传**元组**：`np.zeros((2, 3))` |
| `np.array([1, 2.5])` | 变成 float 数组 | 数组元素类型必须统一，会往"宽"的靠 |
| `np.empty` 期望是全 0 | 里面是随机垃圾值 | 它不清零，用 `np.zeros` |
| `arange` 做浮点步长 | 少一个数 / 多一个数 | 浮点等分用 `linspace` |
| `linspace(a, b, n)` 的 n 当步长 | 结果数量不对 | 第三个参数是**个数**，不是步长 |
| `arr.view()` 当成复制 | 改一个另一个也变了 | 要独立副本用 `copy()` |
| 忘了设随机种子 | 每次跑结果不同 | `np.random.seed(42)` |
| `arr * 2` 以为会像 list 那样重复 | 其实是每个元素乘 2 | 记住"元素级运算"这个核心语义 |
| `dtype=int` 塞浮点数 | 小数被截断 | 按需选 `float` |
| 把 Python list 当数组用 | 运算结果不符合预期 | 先 `np.array(...)` 转过来 |

---

## 九、动手练习

**练习 1**：创建一个 3 行 4 列、值全为 7 的整数数组。

**练习 2**：用 `linspace` 生成 0 到 100 之间等间隔的 11 个数。

**练习 3**：生成 5 行 5 列、范围在 [1, 100] 的随机整数数组，
要求结果可复现（每次运行都一样）。

**练习 4**：验证 `arange(0, 10, 3)` 和 `linspace(0, 10, 4)` 的结果差异，
解释为什么（提示：一个按步长，一个按个数）。

**练习 5**：创建一个数组 `a`，用 `view()` 和 `copy()` 分别得到 `v`、`c`，
然后改 `a[0]`，观察 `v` 和 `c` 的变化并解释。

**练习 6**：生成一个 3 行 3 列的对角矩阵，对角线元素是 `[10, 20, 30]`。

---

**答案思路（先自己做）**

```python
# 1
print(np.full((3, 4), 7, dtype=int))

# 2
print(np.linspace(0, 100, 11))

# 3
np.random.seed(1)
print(np.random.randint(1, 101, (5, 5)))

# 6
print(np.diag([10, 20, 30]))
```
